# VisionServe — one-time YOLO26-nano fine-tune (free Colab GPU)

This is the **only** GPU step in the whole project. Runtime → Change runtime type → **T4 GPU**.

Fine-tunes `yolo26n` on the PPE dataset (6 classes: Gloves, Vest, goggles, helmet, mask, safety_shoe).
When done, download `best.pt`, drop it into the repo as `model/best.pt`, and re-run the CPU pipeline
(`model/export.py`, `model/benchmark.py`) against it. Until then the repo uses the pre-trained
fallback `model/ppe_yolov8n.pt` (same dataset, same classes).

In [ ]:
%pip install -q ultralytics

In [ ]:
# dataset: PPE Detection_DATA v3 (CC BY 4.0), YOLO format with train/valid/test splits
!curl -sL -o /content/ppe.zip "https://huggingface.co/Tanishjain9/yolov8n-ppe-detection-6classes/resolve/main/PPE%20Detection_DATA.v3-ppe_dataset_6-class.yolov8.zip"
!unzip -q -o /content/ppe.zip -d /content/ppe

yaml_text = """
path: /content/ppe
train: train/images
val: valid/images
test: test/images
nc: 6
names: ['Gloves', 'Vest', 'goggles', 'helmet', 'mask', 'safety_shoe']
"""
open("/content/ppe.yaml", "w").write(yaml_text)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
model.train(data="/content/ppe.yaml", epochs=50, imgsz=640)

In [ ]:
# honest check on the held-out test split, then download the weights
best = YOLO("runs/detect/train/weights/best.pt")
metrics = best.val(data="/content/ppe.yaml", split="test")
print("test mAP@0.5:", metrics.box.map50, " mAP@0.5:0.95:", metrics.box.map)

from google.colab import files
files.download("runs/detect/train/weights/best.pt")